# Model input preparation: 2019 and 2025

This notebook reproduces two model-input CSVs from the existing cleaned data. It retains field definitions, preparation rules, observed results, alternatives, and limitations. It does not fit a model, standardize features, or assign low/medium/high risk.

## 0. Inputs, environment, and record eligibility

Run all cells in order in a fresh Python kernel. Keep this notebook in the repository, or set `PROJECT_ROOT` in the next cell. Required packages are pandas, NumPy, and IPython; JupyterLab/ipykernel are needed to run the notebook. The verified environment uses Python 3.12, pandas 3.1.0, NumPy 2.5.4, and ipykernel 7.4.0. Version-specific input assertions intentionally stop execution if a different source release is supplied.

This notebook starts from the existing **cleaned** tables, not directly from the full raw-data cleaning workflow. The following repository files are required:

- `DataCleaningFeatureEngeering/ProposalModelDataProcessing/deliverables/data_handoff_review_20261031/data/cleaned_2019.csv`
- `DataCleaningFeatureEngeering/ProposalModelDataProcessing/deliverables/data_handoff_review_20261031/data/cleaned_2025.csv`
- `raw_data/2019_Food_Access_Research_Atlas_Data/Food Access Research Atlas.csv`
- `raw_data/2025-sram-fara-data/SRAM Straight Line Distance Data.csv`

The raw files are used to check whether missing spatial fields can be recovered. Driving-distance data are not used. Source files are never overwritten. The export section writes only `model_input_2019.csv` and `model_input_2025.csv` to a new timestamped directory. No per-record logs, extra reports, or metadata files are produced.

2019 and 2025 stay in separate DataFrames. Eligible records require positive population, a known official `FD_2`, and complete children, seniors, and seven race/ethnicity counts.

- **2019:** 72,531 source rows minus four records with jointly missing demographic composition = **72,527 rows**. Their FD_2 labels are known; exclusion is due to incomplete selected inputs, not unknown targets. Independent imputation of all composition fields could break their total-population relationship.
- **2025:** 84,119 source rows minus 271 zero-population records = **83,848 rows**. Those 271 records also lack FD_2. They are not assigned population, a target, or a low-risk label.

These are overlapping field-level issues within the same excluded records, not counts to add across columns. All other eligible records are retained. Filtering applies only to model inputs; the complete source tables remain unchanged.

### Target and feature boundaries

`FD_2` carries the official low-access flag: 2019 `LA1and20` and its corresponding 2025 straight-line field. Its threshold is at least 500 people **or** 33% of the tract population beyond 1 mile in urban areas / 20 miles in rural areas. It does not add a low-income requirement. The project uses this as its FD target; it is not a calibrated risk probability or the combined low-income/low-access definition. Never impute the target or derive it from imputed population counts.

The export contains 25 candidate features, an ID, and the target. A 21-feature option excludes the three spatial flags and low-access population. Exact feature lists are defined in Section 23. IDs, the target, QA, and imputation indicators must not be added to X automatically; clustering also excludes the target.

Some 2025 imputations use 2025 observed feature distributions. This is disclosed throughout: the preprocessing is **not** wholly fitted on 2019, although target values are never used to select imputation values or donor groups. Store definitions, population reference years, and tract boundaries differ across releases; corresponding field names do not remove those differences.


In [1]:
from pathlib import Path
import hashlib
import pandas as pd
PROJECT_ROOT = None  # Optionally set an absolute repository path.
ROOT = Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT else next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'raw_data').is_dir() and (p / 'DataCleaningFeatureEngeering').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Set PROJECT_ROOT to the repository containing raw_data and DataCleaningFeatureEngeering.')
PACKAGE = ROOT / 'DataCleaningFeatureEngeering/ProposalModelDataProcessing/deliverables/data_handoff_review_20261031'

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
source_paths = {year: PACKAGE / f'data/cleaned_{year}.csv' for year in ['2019', '2025']}
source_hashes = {year: sha256(path) for year, path in source_paths.items()}
masters = {year: pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig') for year, path in source_paths.items()}
composition_fields = ['TractKids', 'TractSeniors', 'TractWhite', 'TractBlack', 'TractAsian', 'TractNHOPI', 'TractAIAN', 'TractOMultir', 'TractHispanic']
eligibility = {}
for year, table in masters.items():
    population = pd.to_numeric(table.clean__population_total, errors='raise')
    complete = table[['clean__' + f for f in composition_fields]].ne('').all(axis=1)
    eligibility[year] = population.gt(0) & table.FD_2_status.eq('known') & complete
    kept = eligibility[year].sum()
    assert table.tract_id.is_unique and table.tract_id.str.fullmatch('[0-9]{11}').all()
    assert table.loc[eligibility[year], 'FD_2'].isin(['0', '1']).all()
    assert kept == {'2019': 72527, '2025': 83848}[year]

from IPython.display import display

def stage_summary(before, after, fields):
    rows = []
    for year in ['2019', '2025']:
        for field in fields:
            rows.append({'year': year, 'field': field, 'eligible_rows': len(after[year]),
                         'missing_before': int(before[year][field].isna().sum()),
                         'missing_after': int(after[year][field].isna().sum())})
    return pd.DataFrame(rows)

display(pd.DataFrame([{'year': y, 'source_rows': len(masters[y]),
    'eligible_rows': int(eligibility[y].sum()), 'excluded_rows': int((~eligibility[y]).sum())}
    for y in ['2019', '2025']]))


,year,source_rows,eligible_rows,excluded_rows
0,2019,72531,72527,4
1,2025,84119,83848,271


## 1. Urban: retain the official value

`clean__Urban`: 0 = rural, 1 = urban. Both years contain valid observed binary values for all eligible records. No imputation, recoding, or rescaling is needed. Urban status determines the distance threshold associated with the low-access population field. Do not infer it from the spatial flags.

## 2. Total population: retain positive observed counts

`clean__population_total` uses Census 2010 population for the 2019 release and Census 2020 population for 2025. Neither source field is missing. The 271 zero-population records in 2025 are excluded by eligibility; zero is not replaced by 1 or a median. Remaining values are retained unchanged.

## 3. Occupied housing units: retain observed values, including zero

`clean__occupied_housing_units_total` represents occupied housing units, not all housing units. Source fields are `OHU2010` / `OHU2020`. Both years have no missing values. Zero counts in the full tables are 106 / 420; eligible zero counts are **106 / 165**.

A tract can have population in group quarters but no ordinary occupied housing units. Do not automatically delete it, impute zero as missing, or change its target. This Census reference is not automatically a suitable same-year denominator for ACS SNAP or vehicle-access counts. Alternatives include using a verified, compatible ACS denominator if a model specifically requires household rates; no such rate conversion is performed here.

## 4. GroupQuartersFlag: retain the official classification

`clean__GroupQuartersFlag` is an existing official binary field: 1 corresponds to a group-quarters population share of at least 67%, and 0 to a lower share. Group quarters include dormitories, military quarters, correctional institutions, and nursing facilities. A value of 0 does not imply zero group-quarters residents.

2019 has no missing flag; the 271 missing flags in 2025 occur in excluded zero-population records. Retain all eligible official flags. The flag does not reconstruct an exact missing count or percentage.


## 5. NUMGQTRS and PCTGQTRS: impute the share and derive the count

`clean__NUMGQTRS` is the number of group-quarters residents. `clean__PCTGQTRS` is their share of total population, in **0-100 percent units**.

2019 has 25 jointly missing count/share pairs in the source; 21 remain eligible, all with flag 0. No eligible 2025 record needs either field filled. Existing official flags can be present even where a separately published count/share is missing; do not use a flag as an exact numerical replacement.

Use observed eligible **2019** shares consistent with their official flag: below 67% for flag 0, at least 67% for flag 1. Select the median within state and flag, then fall back to the national same-flag group. Each reference needs at least five donors; otherwise stop. Do not cross flag groups. Derive count = population × imputed percentage / 100. Retain fractional estimates and all observed values.

**Result:** all 21 pairs use state/same-flag reference, with 72,506 valid donors overall. Imputed shares are 0.13% for seven Arizona records, 0.27% for one California record, 0.28% for 12 New York records, and 0.05% for one Virginia record. No national fallback is used.

**Alternatives:** recover compatible Census observations, use a validated population-constrained joint imputation, or omit both fields from X. Separately imputing count and share may break their relationship. Group medians are estimates, not recovered observations.


In [2]:
from IPython.display import display
MIN_GQ_DONORS = 5
gq_before = {}
for year, table in masters.items():
    cols = ['tract_id', 'State', 'clean__population_total', 'clean__GroupQuartersFlag', 'clean__NUMGQTRS', 'clean__PCTGQTRS']
    part = table.loc[eligibility[year], cols].copy().reset_index(drop=True)
    for col in cols[2:]:
        part[col] = pd.to_numeric(part[col].replace('', pd.NA), errors='raise').astype('Float64')
    assert part.clean__population_total.gt(0).all()
    assert part.clean__GroupQuartersFlag.isin([0, 1]).all()
    part['state_fips'] = part.tract_id.str[:2]
    gq_before[year] = part
train = gq_before['2019']
share = train.clean__PCTGQTRS
flag = train.clean__GroupQuartersFlag
compatible = (flag.eq(0) & share.lt(67) | flag.eq(1) & share.ge(67)).fillna(False)
donor_mask = share.between(0, 100).fillna(False) & compatible
observed_mismatch = share.notna() & ~donor_mask
donors = train.loc[donor_mask]
gq_state_reference = donors.groupby(['state_fips', 'clean__GroupQuartersFlag'])['clean__PCTGQTRS'].agg(['count', 'median'])
gq_national_reference = donors.groupby('clean__GroupQuartersFlag')['clean__PCTGQTRS'].agg(['count', 'median'])
gq_prepared = {}
for year, before in gq_before.items():
    result = before.copy()
    missing = before.clean__PCTGQTRS.isna()
    assert missing.equals(before.clean__NUMGQTRS.isna()), 'Input validation failed: check source values, donor availability, and expected record counts.'
    assert int(missing.sum()) == {'2019': 21, '2025': 0}[year]
    if missing.any():
        assert before.loc[missing, 'clean__GroupQuartersFlag'].eq(0).all()
    for idx, row in before.loc[missing].iterrows():
        state, group = (row.state_fips, row.clean__GroupQuartersFlag)
        key = (state, group)
        reference = gq_state_reference.loc[key] if key in gq_state_reference.index else None
        level = 'state_same_flag'
        if reference is None or reference['count'] < MIN_GQ_DONORS:
            reference = gq_national_reference.loc[group] if group in gq_national_reference.index else None
            level = 'national_same_flag'
        if reference is None or reference['count'] < MIN_GQ_DONORS:
            raise ValueError(f'{year} / {row.tract_id}Input validation failed: check source values, donor availability, and expected record counts.')
        filled_share = float(reference['median'])
        filled_count = float(row.clean__population_total) * filled_share / 100
        result.loc[idx, 'clean__PCTGQTRS'] = filled_share
        result.loc[idx, 'clean__NUMGQTRS'] = filled_count
    assert result[['clean__NUMGQTRS', 'clean__PCTGQTRS']].notna().all().all()
    assert result.clean__PCTGQTRS.between(0, 100).all()
    assert result.clean__NUMGQTRS.ge(0).all()
    pd.testing.assert_frame_equal(result.loc[~missing], before.loc[~missing])
    pd.testing.assert_frame_equal(result.drop(columns=['clean__NUMGQTRS', 'clean__PCTGQTRS']), before.drop(columns=['clean__NUMGQTRS', 'clean__PCTGQTRS']))
    if missing.any():
        assert result.loc[missing, 'clean__PCTGQTRS'].lt(67).all()
        expected_count = result.loc[missing, 'clean__population_total'] * result.loc[missing, 'clean__PCTGQTRS'] / 100
        assert (result.loc[missing, 'clean__NUMGQTRS'] - expected_count).abs().lt(1e-09).all()
    gq_prepared[year] = result
for year, path in source_paths.items():
    assert sha256(path) == source_hashes[year], 'Input validation failed: check source values, donor availability, and expected record counts.'
    observed_missing = masters[year].clean__PCTGQTRS.eq('').sum()
    assert observed_missing == {'2019': 25, '2025': 271}[year], 'Input validation failed: check source values, donor availability, and expected record counts.'

display(stage_summary(gq_before, gq_prepared, ['clean__NUMGQTRS', 'clean__PCTGQTRS']))


,year,field,eligible_rows,missing_before,missing_after
0,2019,clean__NUMGQTRS,72527,21,0
1,2019,clean__PCTGQTRS,72527,21,0
2,2025,clean__NUMGQTRS,83848,0,0
3,2025,clean__PCTGQTRS,83848,0,0


## 6. TractKids: retain observed counts

`clean__TractKids` counts children aged 0-17. Its four missing 2019 values belong to the same excluded demographic records; 2025 has no missing values. Eligible records need no imputation or transformation. Valid zero counts are 144 in 2019 and 74 in 2025 and are retained.

Do not recover this count by subtracting seniors from total population.

**Alternative if the excluded records are needed:** recover observations with matching year, boundaries, and category definitions, or use a jointly constrained composition method. Do not independently fill every population component and silently break the six-race total. No such recovery is performed here.


## 7. TractSeniors: retain observed counts

`clean__TractSeniors` counts people aged 65 and older. Its four missing 2019 values belong to the same excluded demographic records; 2025 has no missing values. Eligible records need no imputation or transformation. Valid zero counts are 187 in 2019 and 169 in 2025 and are retained.

Total population minus children is not the senior population.

**Alternative if the excluded records are needed:** recover observations with matching year, boundaries, and category definitions, or use a jointly constrained composition method. Do not independently fill every population component and silently break the six-race total. No such recovery is performed here.


## 8. TractWhite: retain observed counts

`clean__TractWhite` counts White residents. Its four missing 2019 values belong to the same excluded demographic records; 2025 has no missing values. Eligible records need no imputation or transformation. Valid zero counts are 29 in 2019 and 104 in 2025 and are retained.

This is a count, not a binary indicator or a percentage.

**Alternative if the excluded records are needed:** recover observations with matching year, boundaries, and category definitions, or use a jointly constrained composition method. Do not independently fill every population component and silently break the six-race total. No such recovery is performed here.


## 9. TractBlack: retain observed counts

`clean__TractBlack` counts Black or African American residents. Its four missing 2019 values belong to the same excluded demographic records; 2025 has no missing values. Eligible records need no imputation or transformation. Valid zero counts are 423 in 2019 and 619 in 2025 and are retained.

This is a count, not a binary indicator or a percentage.

**Alternative if the excluded records are needed:** recover observations with matching year, boundaries, and category definitions, or use a jointly constrained composition method. Do not independently fill every population component and silently break the six-race total. No such recovery is performed here.


## 10. TractAsian: retain observed counts

`clean__TractAsian` counts Asian residents. Its four missing 2019 values belong to the same excluded demographic records; 2025 has no missing values. Eligible records need no imputation or transformation. Valid zero counts are 1,146 in 2019 and 1,481 in 2025 and are retained.

This is a count, not a binary indicator or a percentage.

**Alternative if the excluded records are needed:** recover observations with matching year, boundaries, and category definitions, or use a jointly constrained composition method. Do not independently fill every population component and silently break the six-race total. No such recovery is performed here.


## 11. TractNHOPI: retain observed counts

`clean__TractNHOPI` counts Native Hawaiian and Other Pacific Islander residents. Its four missing 2019 values belong to the same excluded demographic records; 2025 has no missing values. Eligible records need no imputation or transformation. Valid zero counts are 27,368 in 2019 and 30,981 in 2025 and are retained.

Many valid zeros do not justify treating zero as missing.

**Alternative if the excluded records are needed:** recover observations with matching year, boundaries, and category definitions, or use a jointly constrained composition method. Do not independently fill every population component and silently break the six-race total. No such recovery is performed here.


## 12. TractAIAN: retain observed counts

`clean__TractAIAN` counts American Indian and Alaska Native residents. Its four missing 2019 values belong to the same excluded demographic records; 2025 has no missing values. Eligible records need no imputation or transformation. Valid zero counts are 1,495 in 2019 and 2,401 in 2025 and are retained.

Preserve the source category; do not infer it from another race.

**Alternative if the excluded records are needed:** recover observations with matching year, boundaries, and category definitions, or use a jointly constrained composition method. Do not independently fill every population component and silently break the six-race total. No such recovery is performed here.


## 13. TractOMultir: retain observed counts

`clean__TractOMultir` counts Other / Multiple race residents. Its four missing 2019 values belong to the same excluded demographic records; 2025 has no missing values. Eligible records need no imputation or transformation. Valid zero counts are 140 in 2019 and 28 in 2025 and are retained.

Retain the combined source category; it is not a residual to be redefined.

**Alternative if the excluded records are needed:** recover observations with matching year, boundaries, and category definitions, or use a jointly constrained composition method. Do not independently fill every population component and silently break the six-race total. No such recovery is performed here.


## 14. TractHispanic: retain observed counts

`clean__TractHispanic` counts Hispanic or Latino residents. Its four missing 2019 values belong to the same excluded demographic records; 2025 has no missing values. Eligible records need no imputation or transformation. Valid zero counts are 129 in 2019 and 68 in 2025 and are retained.

Hispanic is an ethnicity overlapping the race categories. Do not include it in the six-race population sum.

**Alternative if the excluded records are needed:** recover observations with matching year, boundaries, and category definitions, or use a jointly constrained composition method. Do not independently fill every population component and silently break the six-race total. No such recovery is performed here.


## 15. LowIncomeTracts: retain the official flag

`clean__LowIncomeTracts` is the official 0/1 low-income classification, not a low-income headcount or percentage. Eligibility involves a poverty rate of at least 20% or family-income conditions relative to state/metropolitan reference medians (80% criterion); it is not based on one national dollar cutoff.

There are no missing values in either release. In 2019, 29 eligible records display PovertyRate = 20 but LowIncomeTracts = 0; no such combination occurs in 2025. Published precision or construction details may contribute, but the cause is not established. Retain official values and do not overwrite them after poverty or income imputation.

**Alternative:** rebuild the flag only with the full official rule and compatible reference medians/precision, as a separately defined variable. That is not done here.

## 16. PovertyRate: impute from 2019 reference medians

`clean__PovertyRate` is an ACS percentage in 0-100 units: 17.4 means 17.4%, not 0.174. Its denominator is the population for whom poverty status is determined, not automatically the Census total. It is not interchangeable with TractLOWI.

The three missing 2019 values are in excluded records. 2025 has 701 source missing values; 433 remain eligible. Use observed eligible 2019 values including valid zeros. Select state/same-Urban median, then national/same-Urban, then national overall. Each group needs at least five observations; stop if none qualifies. Never group by FD_2 or use filled values as donors.

**Result:** all 433 values use state/same-Urban reference; no national fallback. Existing values and the official LowIncomeTracts flag are unchanged.

**Alternatives:** verified ACS recovery, a training-fitted predictive imputer, or same-year observed-feature medians with explicit test-distribution disclosure. The selected rule uses 2019 reference only.


In [3]:
MIN_POVERTY_DONORS = 5
poverty_before = {}
for year, table in masters.items():
    part = table.loc[eligibility[year], ['tract_id', 'State', 'clean__Urban', 'clean__PovertyRate']].copy().reset_index(drop=True)
    for col in ['clean__Urban', 'clean__PovertyRate']:
        part[col] = pd.to_numeric(part[col].replace('', pd.NA), errors='raise').astype('Float64')
    part['state_fips'] = part.tract_id.str[:2]
    assert part.clean__Urban.isin([0, 1]).all()
    assert part.clean__PovertyRate.dropna().between(0, 100).all()
    poverty_before[year] = part
poverty_donors = poverty_before['2019'].dropna(subset=['clean__PovertyRate'])
poverty_state_reference = poverty_donors.groupby(['state_fips', 'clean__Urban'])['clean__PovertyRate'].agg(['count', 'median'])
poverty_urban_reference = poverty_donors.groupby('clean__Urban')['clean__PovertyRate'].agg(['count', 'median'])
poverty_overall_reference = poverty_donors.clean__PovertyRate.agg(['count', 'median'])
assert len(poverty_donors) == 72527
poverty_prepared = {}
for year, before in poverty_before.items():
    result = before.copy()
    missing = before.clean__PovertyRate.isna()
    assert int(missing.sum()) == {'2019': 0, '2025': 433}[year]
    for idx, row in before.loc[missing].iterrows():
        key = (row.state_fips, row.clean__Urban)
        reference = poverty_state_reference.loc[key] if key in poverty_state_reference.index else None
        level = 'state_same_urban'
        if reference is None or reference['count'] < MIN_POVERTY_DONORS:
            reference = poverty_urban_reference.loc[row.clean__Urban] if row.clean__Urban in poverty_urban_reference.index else None
            level = 'national_same_urban'
        if reference is None or reference['count'] < MIN_POVERTY_DONORS:
            reference = poverty_overall_reference
            level = 'national_all'
        if reference['count'] < MIN_POVERTY_DONORS:
            raise ValueError(f'{year}/{row.tract_id}Input validation failed: check source values, donor availability, and expected record counts.')
        fill_value = float(reference['median'])
        result.loc[idx, 'clean__PovertyRate'] = fill_value
    assert result.clean__PovertyRate.notna().all()
    assert result.clean__PovertyRate.between(0, 100).all()
    pd.testing.assert_frame_equal(result.loc[~missing], before.loc[~missing])
    pd.testing.assert_frame_equal(result.drop(columns='clean__PovertyRate'), before.drop(columns='clean__PovertyRate'))
    poverty_prepared[year] = result
for year, path in source_paths.items():
    assert sha256(path) == source_hashes[year], 'Input validation failed: check source values, donor availability, and expected record counts.'
    assert masters[year].clean__PovertyRate.eq('').sum() == {'2019': 3, '2025': 701}[year]

display(stage_summary(poverty_before, poverty_prepared, ['clean__PovertyRate']))


,year,field,eligible_rows,missing_before,missing_after
0,2019,clean__PovertyRate,72527,0,0
1,2025,clean__PovertyRate,83848,433,0


## 17. Family income: preserve price-adjusted values and fill genuine missingness

Use `income_with_endpoint_proxy_2018_usd`, already expressed in **2018 dollars** in the cleaned inputs. Do not apply price conversion again. This preparation step preserves the upstream conversion and endpoint proxy values; it does not recompute them from raw income. Raw endpoint codes 2499 / 250001 represent lower/upper reporting limits, not exact observed incomes.

Source missing income: 748 / 2,143 in 2019 / 2025; eligible missing: **745 / 1,873**. Eligible ordinary observations: 71,510 / 80,330. Eligible endpoint proxies: 272 / 1,645; these already have amounts and are not missing-value targets.

Only eligible **2019 ordinary observed income**, from `income_ordinary_2018_usd` with `income_value_status == ordinary`, supplies reference values. Exclude endpoints, missing values, and previously imputed amounts. Use county → state → national for 2019. For 2025, use the 2019 state → national references; no verified cross-year county mapping is assumed. Require at least five donors at each level. County/state keys use the first five/two tract-ID digits. These are medians of tract median incomes, not official county household-income estimates.

**Result:** 2019 uses county medians for 721 records and state fallback for 24. All 1,873 missing 2025 values use 2019 state medians. No national fallback. Existing income and endpoint proxy amounts remain unchanged.

**Alternatives:** matched ACS recovery, simpler state/national medians, training-fitted predictive or multiple imputation. Endpoint modeling or exclusion of income is also possible, but requires an explicit feature/model decision. No method comparison or model fitting is performed here.


In [4]:
import math
MIN_INCOME_DONORS = 5
INCOME_COLUMN = 'income_with_endpoint_proxy_2018_usd'
income_before = {}
for year, table in masters.items():
    cols = ['tract_id', 'State', 'income_value_status', 'income_ordinary_2018_usd', INCOME_COLUMN]
    part = table.loc[eligibility[year], cols].copy().reset_index(drop=True)
    for col in ['income_ordinary_2018_usd', INCOME_COLUMN]:
        part[col] = pd.to_numeric(part[col].replace('', pd.NA), errors='raise').astype('Float64')
    part['state_fips'] = part.tract_id.str[:2]
    part['county_fips'] = part.tract_id.str[:5]
    assert part.income_value_status.isin(['ordinary', 'lower_endpoint', 'upper_endpoint', 'source_missing']).all()
    assert part[INCOME_COLUMN].isna().equals(part.income_value_status.eq('source_missing'))
    for value in part[INCOME_COLUMN].dropna():
        assert math.isfinite(value) and value > 0
    income_before[year] = part
income_donors = income_before['2019'].loc[income_before['2019'].income_value_status.eq('ordinary')].copy()
assert len(income_donors) == 71510
assert income_donors.income_ordinary_2018_usd.notna().all()
assert income_donors.income_ordinary_2018_usd.map(lambda value: math.isfinite(value) and value > 0).all()
income_county_reference = income_donors.groupby('county_fips')['income_ordinary_2018_usd'].agg(['count', 'median'])
income_state_reference = income_donors.groupby('state_fips')['income_ordinary_2018_usd'].agg(['count', 'median'])
income_national_reference = income_donors.income_ordinary_2018_usd.agg(['count', 'median'])
income_prepared = {}
for year, before in income_before.items():
    result = before.copy()
    missing = before[INCOME_COLUMN].isna()
    assert int(missing.sum()) == {'2019': 745, '2025': 1873}[year]
    for idx, row in before.loc[missing].iterrows():
        reference = None
        level = 'county'
        reference_key = row.county_fips
        if year == '2019' and row.county_fips in income_county_reference.index:
            reference = income_county_reference.loc[row.county_fips]
        if reference is None or reference['count'] < MIN_INCOME_DONORS:
            reference = income_state_reference.loc[row.state_fips] if row.state_fips in income_state_reference.index else None
            level, reference_key = ('state', row.state_fips)
        if reference is None or reference['count'] < MIN_INCOME_DONORS:
            reference = income_national_reference
            level, reference_key = ('national', 'ALL')
        if reference['count'] < MIN_INCOME_DONORS:
            raise ValueError(f'{year}/{row.tract_id}Input validation failed: check source values, donor availability, and expected record counts.')
        value = float(reference['median'])
        assert math.isfinite(value) and value > 0
        result.loc[idx, INCOME_COLUMN] = value
    assert result[INCOME_COLUMN].notna().all()
    assert result[INCOME_COLUMN].map(lambda value: math.isfinite(value) and value > 0).all()
    pd.testing.assert_frame_equal(result.loc[~missing], before.loc[~missing])
    pd.testing.assert_frame_equal(result.drop(columns=INCOME_COLUMN), before.drop(columns=INCOME_COLUMN))
    endpoints = before.income_value_status.isin(['lower_endpoint', 'upper_endpoint'])
    assert int(endpoints.sum()) == {'2019': 272, '2025': 1645}[year]
    pd.testing.assert_series_equal(result.loc[endpoints, INCOME_COLUMN], before.loc[endpoints, INCOME_COLUMN])
    income_prepared[year] = result
for year in ['2019', '2025']:
    pass
for year, path in source_paths.items():
    assert sha256(path) == source_hashes[year], 'Input validation failed: check source values, donor availability, and expected record counts.'
    assert masters[year][INCOME_COLUMN].eq('').sum() == {'2019': 748, '2025': 2143}[year]

display(stage_summary(income_before, income_prepared, ['income_with_endpoint_proxy_2018_usd']))


,year,field,eligible_rows,missing_before,missing_after
0,2019,income_with_endpoint_proxy_2018_usd,72527,745,0
1,2025,income_with_endpoint_proxy_2018_usd,83848,1873,0


## 18. TractLOWI: same-year grouped median imputation

`clean__TractLOWI` represents ACS low-income persons, using the applicable 200% poverty-threshold criterion; not the poverty-rate denominator or a household count. The four source missing values in 2019 are excluded by demographic eligibility; none remains. All **14 missing 2025 values** remain eligible. They are the same tracts across LOWI, HUNV, and SNAP, not 42 different tracts.

Use original observed **2025** values, including valid zeros: state/same-Urban → national/same-Urban → national overall. Require at least five donors at each level and stop if none qualifies. Calculate each field's references separately. Only replace NA; retain fractional estimates, observed values, and existing cross-period values above Census reference totals. Do not use FD_2 or filled values as donors. Do not backfill 2019 with 2025 observations.

**Result:** all 14 use the New York urban group of 4,474 observed records. Each is filled with **2,360 people**; no fallback. This uses test-year feature distributions and must not be described as train-only preprocessing.

**Scale limitation:** tract `36103201200` has a Census reference total of 560 people, below the imputed count. Retain this limitation rather than silently clipping. Cross-period differences do not prove that this estimate is appropriate. A fractional household count is a statistical estimate, not an observed fraction of a household.

**Alternatives:** use 2019 grouped references (which would give 964 people for these records), recover compatible official observations, or use a validated scale-aware method. Changing reference year alone does not solve small-tract scale mismatch. No alternate method is executed.


In [5]:
MIN_LOWI_DONORS = 5
lowi_before = {}
for year, table in masters.items():
    part = table.loc[eligibility[year], ['tract_id', 'State', 'clean__Urban', 'clean__population_total', 'clean__TractLOWI']].copy().reset_index(drop=True)
    for col in ['clean__Urban', 'clean__population_total', 'clean__TractLOWI']:
        part[col] = pd.to_numeric(part[col].replace('', pd.NA), errors='raise').astype('Float64')
    part['state_fips'] = part.tract_id.str[:2]
    assert part.clean__Urban.isin([0, 1]).all()
    assert part.clean__TractLOWI.dropna().map(lambda value: math.isfinite(value) and value >= 0).all()
    lowi_before[year] = part
lowi_donors = lowi_before['2025'].dropna(subset=['clean__TractLOWI'])
assert len(lowi_donors) == 83834
lowi_state_reference = lowi_donors.groupby(['state_fips', 'clean__Urban'])['clean__TractLOWI'].agg(['count', 'median'])
lowi_urban_reference = lowi_donors.groupby('clean__Urban')['clean__TractLOWI'].agg(['count', 'median'])
lowi_overall_reference = lowi_donors.clean__TractLOWI.agg(['count', 'median'])
lowi_prepared = {}
for year, before in lowi_before.items():
    result = before.copy()
    missing = before.clean__TractLOWI.isna()
    assert int(missing.sum()) == {'2019': 0, '2025': 14}[year]
    assert year == '2025' or not missing.any(), 'Input validation failed: check source values, donor availability, and expected record counts.'
    for idx, row in before.loc[missing].iterrows():
        key = (row.state_fips, row.clean__Urban)
        reference = lowi_state_reference.loc[key] if key in lowi_state_reference.index else None
        level = 'state_same_urban'
        if reference is None or reference['count'] < MIN_LOWI_DONORS:
            reference = lowi_urban_reference.loc[row.clean__Urban] if row.clean__Urban in lowi_urban_reference.index else None
            level = 'national_same_urban'
        if reference is None or reference['count'] < MIN_LOWI_DONORS:
            reference, level = (lowi_overall_reference, 'national_all')
        if reference['count'] < MIN_LOWI_DONORS:
            raise ValueError(f'{year}/{row.tract_id}Input validation failed: check source values, donor availability, and expected record counts.')
        value = float(reference['median'])
        result.loc[idx, 'clean__TractLOWI'] = value
    assert result.clean__TractLOWI.notna().all()
    assert result.clean__TractLOWI.map(lambda value: math.isfinite(value) and value >= 0).all()
    pd.testing.assert_frame_equal(result.loc[~missing], before.loc[~missing])
    pd.testing.assert_frame_equal(result.drop(columns='clean__TractLOWI'), before.drop(columns='clean__TractLOWI'))
    lowi_prepared[year] = result
for year, path in source_paths.items():
    assert sha256(path) == source_hashes[year], 'Input validation failed: check source values, donor availability, and expected record counts.'
    assert masters[year].clean__TractLOWI.eq('').sum() == {'2019': 4, '2025': 14}[year]

display(stage_summary(lowi_before, lowi_prepared, ['clean__TractLOWI']))


,year,field,eligible_rows,missing_before,missing_after
0,2019,clean__TractLOWI,72527,0,0
1,2025,clean__TractLOWI,83848,14,0


## 19. TractHUNV: same-year grouped median imputation

`clean__TractHUNV` represents ACS households / occupied housing units without vehicle access; not people and not only households far from stores. The four source missing values in 2019 are excluded by demographic eligibility; none remains. All **14 missing 2025 values** remain eligible. They are the same tracts across LOWI, HUNV, and SNAP, not 42 different tracts.

Use original observed **2025** values, including valid zeros: state/same-Urban → national/same-Urban → national overall. Require at least five donors at each level and stop if none qualifies. Calculate each field's references separately. Only replace NA; retain fractional estimates, observed values, and existing cross-period values above Census reference totals. Do not use FD_2 or filled values as donors. Do not backfill 2019 with 2025 observations.

**Result:** all 14 use the New York urban group of 4,474 observed records. Each is filled with **227.5 households**; no fallback. This uses test-year feature distributions and must not be described as train-only preprocessing.

**Scale limitation:** tract `36103201200` has a Census reference total of 25 occupied housing units, below the imputed count. Retain this limitation rather than silently clipping. Cross-period differences do not prove that this estimate is appropriate. A fractional household count is a statistical estimate, not an observed fraction of a household.

**Alternatives:** use 2019 grouped references (which would give 235 households for these records), recover compatible official observations, or use a validated scale-aware method. Changing reference year alone does not solve small-tract scale mismatch. No alternate method is executed.


In [6]:
MIN_HUNV_DONORS = 5
hunv_before = {}
for year, table in masters.items():
    part = table.loc[eligibility[year], ['tract_id', 'State', 'clean__Urban', 'clean__occupied_housing_units_total', 'clean__TractHUNV']].copy().reset_index(drop=True)
    for col in ['clean__Urban', 'clean__occupied_housing_units_total', 'clean__TractHUNV']:
        part[col] = pd.to_numeric(part[col].replace('', pd.NA), errors='raise').astype('Float64')
    part['state_fips'] = part.tract_id.str[:2]
    assert part.clean__Urban.isin([0, 1]).all()
    assert part.clean__TractHUNV.dropna().map(lambda value: math.isfinite(value) and value >= 0).all()
    hunv_before[year] = part
hunv_donors = hunv_before['2025'].dropna(subset=['clean__TractHUNV'])
assert len(hunv_donors) == 83834
hunv_state_reference = hunv_donors.groupby(['state_fips', 'clean__Urban'])['clean__TractHUNV'].agg(['count', 'median'])
hunv_urban_reference = hunv_donors.groupby('clean__Urban')['clean__TractHUNV'].agg(['count', 'median'])
hunv_overall_reference = hunv_donors.clean__TractHUNV.agg(['count', 'median'])
hunv_prepared = {}
for year, before in hunv_before.items():
    result = before.copy()
    missing = before.clean__TractHUNV.isna()
    assert int(missing.sum()) == {'2019': 0, '2025': 14}[year]
    assert year == '2025' or not missing.any(), 'Input validation failed: check source values, donor availability, and expected record counts.'
    for idx, row in before.loc[missing].iterrows():
        key = (row.state_fips, row.clean__Urban)
        reference = hunv_state_reference.loc[key] if key in hunv_state_reference.index else None
        level = 'state_same_urban'
        if reference is None or reference['count'] < MIN_HUNV_DONORS:
            reference = hunv_urban_reference.loc[row.clean__Urban] if row.clean__Urban in hunv_urban_reference.index else None
            level = 'national_same_urban'
        if reference is None or reference['count'] < MIN_HUNV_DONORS:
            reference, level = (hunv_overall_reference, 'national_all')
        if reference['count'] < MIN_HUNV_DONORS:
            raise ValueError(f'{year}/{row.tract_id}Input validation failed: check source values, donor availability, and expected record counts.')
        value = float(reference['median'])
        result.loc[idx, 'clean__TractHUNV'] = value
    assert result.clean__TractHUNV.notna().all()
    assert result.clean__TractHUNV.map(lambda value: math.isfinite(value) and value >= 0).all()
    pd.testing.assert_frame_equal(result.loc[~missing], before.loc[~missing])
    pd.testing.assert_frame_equal(result.drop(columns='clean__TractHUNV'), before.drop(columns='clean__TractHUNV'))
    hunv_prepared[year] = result
for year, path in source_paths.items():
    assert sha256(path) == source_hashes[year], 'Input validation failed: check source values, donor availability, and expected record counts.'
    assert masters[year].clean__TractHUNV.eq('').sum() == {'2019': 4, '2025': 14}[year]

display(stage_summary(hunv_before, hunv_prepared, ['clean__TractHUNV']))


,year,field,eligible_rows,missing_before,missing_after
0,2019,clean__TractHUNV,72527,0,0
1,2025,clean__TractHUNV,83848,14,0


## 20. TractSNAP: same-year grouped median imputation

`clean__TractSNAP` represents households / occupied housing units receiving SNAP; not recipients as people, benefit dollars, or only households far from stores. The four source missing values in 2019 are excluded by demographic eligibility; none remains. All **14 missing 2025 values** remain eligible. They are the same tracts across LOWI, HUNV, and SNAP, not 42 different tracts.

Use original observed **2025** values, including valid zeros: state/same-Urban → national/same-Urban → national overall. Require at least five donors at each level and stop if none qualifies. Calculate each field's references separately. Only replace NA; retain fractional estimates, observed values, and existing cross-period values above Census reference totals. Do not use FD_2 or filled values as donors. Do not backfill 2019 with 2025 observations.

**Result:** all 14 use the New York urban group of 4,474 observed records. Each is filled with **155 households**; no fallback. This uses test-year feature distributions and must not be described as train-only preprocessing.

**Scale limitation:** tract `36103201200` has a Census reference total of 25 occupied housing units, below the imputed count. Retain this limitation rather than silently clipping. Cross-period differences do not prove that this estimate is appropriate. A fractional household count is a statistical estimate, not an observed fraction of a household.

**Alternatives:** use 2019 grouped references (which would give 144 households for these records), recover compatible official observations, or use a validated scale-aware method. Changing reference year alone does not solve small-tract scale mismatch. No alternate method is executed.


In [7]:
MIN_SNAP_DONORS = 5
snap_before = {}
for year, table in masters.items():
    part = table.loc[eligibility[year], ['tract_id', 'State', 'clean__Urban', 'clean__occupied_housing_units_total', 'clean__TractSNAP']].copy().reset_index(drop=True)
    for col in ['clean__Urban', 'clean__occupied_housing_units_total', 'clean__TractSNAP']:
        part[col] = pd.to_numeric(part[col].replace('', pd.NA), errors='raise').astype('Float64')
    part['state_fips'] = part.tract_id.str[:2]
    assert part.clean__Urban.isin([0, 1]).all()
    assert part.clean__TractSNAP.dropna().map(lambda value: math.isfinite(value) and value >= 0).all()
    snap_before[year] = part
snap_donors = snap_before['2025'].dropna(subset=['clean__TractSNAP'])
assert len(snap_donors) == 83834
snap_state_reference = snap_donors.groupby(['state_fips', 'clean__Urban'])['clean__TractSNAP'].agg(['count', 'median'])
snap_urban_reference = snap_donors.groupby('clean__Urban')['clean__TractSNAP'].agg(['count', 'median'])
snap_overall_reference = snap_donors.clean__TractSNAP.agg(['count', 'median'])
snap_prepared = {}
for year, before in snap_before.items():
    result = before.copy()
    missing = before.clean__TractSNAP.isna()
    assert year == '2025' or not missing.any(), 'Input validation failed: check source values, donor availability, and expected record counts.'
    assert int(missing.sum()) == {'2019': 0, '2025': 14}[year]
    for idx, row in before.loc[missing].iterrows():
        key = (row.state_fips, row.clean__Urban)
        reference = snap_state_reference.loc[key] if key in snap_state_reference.index else None
        level = 'state_same_urban'
        if reference is None or reference['count'] < MIN_SNAP_DONORS:
            reference = snap_urban_reference.loc[row.clean__Urban] if row.clean__Urban in snap_urban_reference.index else None
            level = 'national_same_urban'
        if reference is None or reference['count'] < MIN_SNAP_DONORS:
            reference, level = (snap_overall_reference, 'national_all')
        if reference['count'] < MIN_SNAP_DONORS:
            raise ValueError(f'{year}/{row.tract_id}Input validation failed: check source values, donor availability, and expected record counts.')
        value = float(reference['median'])
        result.loc[idx, 'clean__TractSNAP'] = value
    assert result.clean__TractSNAP.notna().all()
    assert result.clean__TractSNAP.map(lambda value: math.isfinite(value) and value >= 0).all()
    pd.testing.assert_frame_equal(result.loc[~missing], before.loc[~missing])
    pd.testing.assert_frame_equal(result.drop(columns='clean__TractSNAP'), before.drop(columns='clean__TractSNAP'))
    snap_prepared[year] = result
for year, path in source_paths.items():
    assert sha256(path) == source_hashes[year], 'Input validation failed: check source values, donor availability, and expected record counts.'
    assert masters[year].clean__TractSNAP.eq('').sum() == {'2019': 4, '2025': 14}[year]

display(stage_summary(snap_before, snap_prepared, ['clean__TractSNAP']))


,year,field,eligible_rows,missing_before,missing_after
0,2019,clean__TractSNAP,72527,0,0
1,2025,clean__TractSNAP,83848,14,0


## 21. Not using a column is a valid preparation option

Keeping the original value/NA and omitting the column from X is an explicit option, not unfinished cleaning. The 21-feature option omits the three spatial flags and low-access population. Do not drop complete tracts just because an unused source field has NA, and do not apply `dropna()` to the full source table.

FD_2 is a separate official target. The earlier formula-based `fd_label` and its status are not substituted for FD_2 or included in X. Original text, QA fields, release information, IDs, and income status remain available in source data for interpretation; they are not automatically predictors.

The next section prepares complete values for the four optional fields because the 25-feature candidate dataset includes them. Providing complete candidates is not a claim that every model should use all of them.

## 22. Three spatial flags and low-access population

### Meaning and retention rules

- `clean__LATracts_half`: official urban half-mile low-access flag. 2019 has no missing values; 2025 has 164 source missing values, all excluded by eligibility. Retain eligible 0/1 values.
- `clean__LATracts1`: official urban one-mile low-access flag. The same missing counts apply; retain eligible 0/1 values.
- `clean__LATractsVehicle_20`: vehicle-access/distance flag, applicable regardless of urban/rural status. 2019 has no missing values; of 121 source missing values in 2025, 14 remain eligible.

The first two flags identify the relevant urban low-access condition using at least 500 people or 33% of population beyond the specified threshold. A rural value of 0 is not evidence of good access. None of these columns is a measured distance or a continuous population share; **NA is not 0**.

### Vehicle flag: attempt recovery, then same-year mode

The official condition is an OR: at least 100 no-vehicle households beyond half a mile; or at least 500 people beyond 20 miles; or at least 33% of population beyond 20 miles. Any known true branch establishes 1; all branches must be known false to establish 0.

For the 14 missing 2025 records, the half-mile no-vehicle count is missing and the 20-mile population is zero. This does not establish a false overall condition. Existing local source components cannot recover the flag. Use original eligible 2025 0/1 observations: state/same-Urban mode → national/same-Urban → national overall. Require at least five donors and a unique mode; a tie triggers fallback, not an arbitrary zero.

**Result:** all 14 become 0 using 4,474 New York urban observations: 4,308 zeros and 166 ones. These are modal estimates, not recovered official values. This uses the 2025 feature distribution.

### Low-access population: same-year median proportion times population

`clean__low_access_population_1_20` counts people beyond 1 mile in urban tracts or 20 miles in rural tracts. 2019 has 35,914 source missing values, of which **35,910 (about 49.51% of eligible records)** require estimation. 2025 has no missing values and is retained unchanged.

Check matching `lapop1` / `lapop20` counts and shares first. None provides a value for these missing 2019 records in the available local files. This is not a claim that every possible external source was searched.

From the 36,617 original observed eligible 2019 records, calculate count / Census population. Select the state/same-Urban median proportion, falling back to national/same-Urban then national overall, with at least five donors. Multiply by the missing tract's own population. Preserve fractional estimates. No 2025 values or FD_2 grouping are used, and no filled value becomes a donor.

**Result:** 29,214 estimates use state/same-Urban reference; 6,696 use national/same-Urban fallback; none uses national overall. Estimates lie between zero and total population. This respects tract size but does not reconstruct true access or solve non-random missingness.

### Important notes for feature selection and alternative methods

- **Definition overlap:** LATracts1 and the urban/rural low-access count directly overlap the official FD_2 definition; the other access flags describe closely related concepts. Using them can reproduce contemporaneous classification rather than demonstrate future-risk prediction.
- **Large imputation burden:** roughly half of eligible 2019 low-access counts are estimated. Observed tracts may not represent missing tracts; medians reduce within-group variation. Numeric completeness is not evidence of information quality.
- **Observed inconsistency after estimation:** 18,641 of 35,910 imputed counts disagree with official FD_2 when the 500-person / 33% formula is applied. Seven of 36,617 observed 2019 counts also disagree; their cause is not assumed to be rounding here. Observed 2025 counts have zero such disagreements. The comparison is diagnostic only: neither estimates nor official targets are altered to force agreement.
- **Valid alternative:** keep source values and NA, omit the affected column(s) from X, and retain the tract. The original 21-feature option does this for all four fields. This may be preferable to treating heavily imputed access counts as equivalent to observations.
- Other alternatives are compatible official-source recovery, 2019-only references for vehicle flags, missing-aware modeling, or a validated scale-aware estimator. Reconstructing counts from FD_2 is not valid recovery. Deleting all missing-count rows would substantially change coverage and is not the selected rule.

Official definitions: [LRAM reference guide](https://www.ers.usda.gov/data-products/food-access-research-atlas/documentation/large-retailer-access-map-reference-guide), [SRAM reference guide](https://www.ers.usda.gov/data-products/food-access-research-atlas/documentation/snap-authorized-retailer-access-map-reference-guide), and the two local source dictionaries.


In [8]:
import numpy as np
SPATIAL_FIELDS = ['clean__LATracts_half', 'clean__LATracts1', 'clean__LATractsVehicle_20', 'clean__low_access_population_1_20']
MIN_SPATIAL_DONORS = 5
spatial_before, spatial_prepared, spatial_recovery_checks = ({}, {}, [])
raw_specs = {'2019': ('2019_Food_Access_Research_Atlas_Data/Food Access Research Atlas.csv', 'utf-8', 'CensusTract', ''), '2025': ('2025-sram-fara-data/SRAM Straight Line Distance Data.csv', 'cp1252', 'CensusTract20', 'SD_SRAM_')}
spatial_raw_hashes = {}
for year, table in masters.items():
    base = table.loc[eligibility[year], ['tract_id', 'clean__Urban', 'clean__population_total'] + SPATIAL_FIELDS].copy().set_index('tract_id')
    for col in base:
        base[col] = pd.to_numeric(base[col].replace('', pd.NA), errors='raise').astype('Float64')
    base['state_fips'] = base.index.str[:2]
    assert base.clean__population_total.gt(0).all()
    for col in SPATIAL_FIELDS[:3]:
        assert base[col].dropna().isin([0, 1]).all()
    spatial_before[year] = base.copy()
    result = base.copy()
    fn, encoding, key, prefix = raw_specs[year]
    path = ROOT / 'raw_data' / fn
    spatial_raw_hashes[path] = sha256(path)
    raw = pd.read_csv(path, dtype=str, encoding=encoding)
    raw.index = raw[key].str.zfill(11)
    assert raw.index.is_unique and base.index.isin(raw.index).all()
    raw = raw.loc[base.index]

    def source_number(name):
        values = raw[prefix + name].replace(['NULL', 'NA', ''], pd.NA)
        return pd.to_numeric(values, errors='raise').astype('Float64')
    for field, original in zip(SPATIAL_FIELDS, ['LATracts_half', 'LATracts1', 'LATractsVehicle_20', 'LAPOP1_20']):
        pd.testing.assert_series_equal(base[field], source_number(original), check_names=False)
    pop = base.clean__population_total
    is_urban = base.clean__Urban.eq(1)
    component_count = source_number('lapop1').where(is_urban, source_number('lapop20'))
    component_share = source_number('lapop1share').where(is_urban, source_number('lapop20share'))
    count_field = SPATIAL_FIELDS[3]
    count_missing = base[count_field].isna()
    vehicle_field = SPATIAL_FIELDS[2]
    vehicle_missing = base[vehicle_field].isna()
    no_vehicle_half = source_number('lahunvhalf')
    population_20 = source_number('lapop20')
    vehicle_true = no_vehicle_half.ge(100).fillna(False) | population_20.ge(500).fillna(False) | (population_20 / pop).ge(0.33).fillna(False)
    vehicle_false = no_vehicle_half.notna() & population_20.notna() & ~vehicle_true
    spatial_recovery_checks.append({'year': year, 'missing_population': int(count_missing.sum()), 'available_matching_count': int(component_count[count_missing].notna().sum()), 'available_matching_share': int(component_share[count_missing].notna().sum()), 'missing_vehicle_flag': int(vehicle_missing.sum()), 'recoverable_vehicle_flag': int(((vehicle_true | vehicle_false) & vehicle_missing).sum())})
    assert not component_count[count_missing].notna().any()
    assert not component_share[count_missing].notna().any()
    assert not ((vehicle_true | vehicle_false) & vehicle_missing).any()
    donor_flag = base.loc[~vehicle_missing]
    for tract, row in base.loc[vehicle_missing].iterrows():
        for level, group in [('state_same_urban', donor_flag.loc[donor_flag.state_fips.eq(row.state_fips) & donor_flag.clean__Urban.eq(row.clean__Urban)]), ('national_same_urban', donor_flag.loc[donor_flag.clean__Urban.eq(row.clean__Urban)]), ('national_all', donor_flag)]:
            values = group[vehicle_field]
            zeros, ones = (int(values.eq(0).sum()), int(values.eq(1).sum()))
            if len(values) >= MIN_SPATIAL_DONORS and zeros != ones:
                value = int(ones > zeros)
                result.loc[tract, vehicle_field] = value
                break
        else:
            raise ValueError(f'{year}/{tract}Input validation failed: check source values, donor availability, and expected record counts.')
    observed = base[count_field].notna()
    shares = base.loc[observed, count_field] / pop.loc[observed]
    assert shares.between(0, 1).all()
    donors = base.loc[observed, ['state_fips', 'clean__Urban']].copy()
    donors['observed_share'] = shares
    state_refs = donors.groupby(['state_fips', 'clean__Urban']).observed_share.agg(['count', 'median'])
    urban_refs = donors.groupby('clean__Urban').observed_share.agg(['count', 'median'])
    all_ref = donors.observed_share.agg(['count', 'median'])
    for tract, row in base.loc[count_missing].iterrows():
        key = (row.state_fips, row.clean__Urban)
        ref = state_refs.loc[key] if key in state_refs.index else None
        level = 'state_same_urban'
        if ref is None or ref['count'] < MIN_SPATIAL_DONORS:
            ref = urban_refs.loc[row.clean__Urban] if row.clean__Urban in urban_refs.index else None
            level = 'national_same_urban'
        if ref is None or ref['count'] < MIN_SPATIAL_DONORS:
            ref, level = (all_ref, 'national_all')
        if ref['count'] < MIN_SPATIAL_DONORS:
            raise ValueError(f'{year}/{tract}Input validation failed: check source values, donor availability, and expected record counts.')
        value = float(ref['median']) * float(row.clean__population_total)
        result.loc[tract, count_field] = value
    for col in SPATIAL_FIELDS:
        assert result[col].notna().all()
        assert result[col].map(math.isfinite).all()
        pd.testing.assert_series_equal(result.loc[base[col].notna(), col], base.loc[base[col].notna(), col])
        result[col + '__imputed'] = base[col].isna().astype('int8')
    for col in SPATIAL_FIELDS[:3]:
        assert result[col].isin([0, 1]).all()
    assert result[count_field].between(0, pop).all()
    pd.testing.assert_frame_equal(result[base.columns.difference(SPATIAL_FIELDS)], base[base.columns.difference(SPATIAL_FIELDS)])
    spatial_prepared[year] = result
spatial_target_diagnostics = []
for year, result in spatial_prepared.items():
    estimated_formula = ((result[SPATIAL_FIELDS[3]] >= 500) | (result[SPATIAL_FIELDS[3]] / result.clean__population_total >= 0.33)).astype(int)
    target = pd.to_numeric(masters[year].set_index('tract_id').loc[result.index, 'FD_2'])
    for label, mask in [('observed_count', ~result[SPATIAL_FIELDS[3] + '__imputed'].astype(bool)), ('imputed_count', result[SPATIAL_FIELDS[3] + '__imputed'].astype(bool))]:
        spatial_target_diagnostics.append({'year': year, 'count_source': label, 'rows': int(mask.sum()), 'formula_disagrees_with_official_FD_2': int(estimated_formula[mask].ne(target[mask]).sum())})
for year, path in source_paths.items():
    assert sha256(path) == source_hashes[year]
for path, original_hash in spatial_raw_hashes.items():
    assert sha256(path) == original_hash

display(stage_summary(spatial_before, spatial_prepared, SPATIAL_FIELDS))
display(pd.DataFrame(spatial_recovery_checks))
display(pd.DataFrame(spatial_target_diagnostics))


,year,field,eligible_rows,missing_before,missing_after
0,2019,clean__LATracts_half,72527,0,0
1,2019,clean__LATracts1,72527,0,0
2,2019,clean__LATractsVehicle_20,72527,0,0
3,2019,clean__low_access_population_1_20,72527,35910,0
4,2025,clean__LATracts_half,83848,0,0
5,2025,clean__LATracts1,83848,0,0
6,2025,clean__LATractsVehicle_20,83848,14,0
7,2025,clean__low_access_population_1_20,83848,0,0


,year,missing_population,available_matching_count,available_matching_share,missing_vehicle_flag,recoverable_vehicle_flag
0,2019,35910,0,0,0,0
1,2025,0,0,0,14,0


,year,count_source,rows,formula_disagrees_with_official_FD_2
0,2019,observed_count,36617,7
1,2019,imputed_count,35910,18641
2,2025,observed_count,83848,0
3,2025,imputed_count,0,0


## 23. Assemble the two feature options and validate alignment

Join all prepared values by the 11-digit string `tract_id`, separately for each year. Preserve source order and all original nonmissing observations. The field names retain their `clean__` prefix, but previously missing cells in the model-input layer now contain estimates; the source tables retain their original NA.

`BASE_FEATURES` has 21 columns; `EXTENDED_FEATURES` adds the four optional columns for a total of 25. `X_base` and `X_extended` hold the corresponding inputs. `y_fd2` holds the independent official target. The checks verify row counts, unique IDs, identical cross-year column order, finite nonnegative values, binary ranges, and population composition.

The six race counts sum to total population. Hispanic overlaps race and is not part of that sum. These dependencies, and the relationship among GQ count/share/flag, may require feature selection for models such as logistic regression. No automatic selection, standardization, train/test split, or model fitting is performed here.

Expected eligible label counts: 2019 has 47,818 zeros and 24,709 ones; 2025 has 70,493 zeros and 13,355 ones. The label is not a low/medium/high risk assignment. Clusters likewise do not inherently have an ordered risk meaning.


In [9]:
BASE_FEATURES = ['clean__Urban', 'clean__population_total', 'clean__occupied_housing_units_total', 'clean__GroupQuartersFlag', 'clean__NUMGQTRS', 'clean__PCTGQTRS', 'clean__TractKids', 'clean__TractSeniors', 'clean__TractWhite', 'clean__TractBlack', 'clean__TractAsian', 'clean__TractNHOPI', 'clean__TractAIAN', 'clean__TractOMultir', 'clean__TractHispanic', 'clean__LowIncomeTracts', 'clean__PovertyRate', 'income_with_endpoint_proxy_2018_usd', 'clean__TractLOWI', 'clean__TractHUNV', 'clean__TractSNAP']
EXTENDED_FEATURES = BASE_FEATURES + SPATIAL_FIELDS
assert len(set(BASE_FEATURES)) == 21 and len(set(EXTENDED_FEATURES)) == 25
X_base, X_extended, y_fd2, model_metadata, imputed_flags = ({}, {}, {}, {}, {})
assembly_summary, field_fill_summary = ([], [])
prepared_sources = [(gq_prepared, ['clean__NUMGQTRS', 'clean__PCTGQTRS']), (poverty_prepared, ['clean__PovertyRate']), (income_prepared, [INCOME_COLUMN]), (lowi_prepared, ['clean__TractLOWI']), (hunv_prepared, ['clean__TractHUNV']), (snap_prepared, ['clean__TractSNAP']), (spatial_prepared, SPATIAL_FIELDS)]
for year, master in masters.items():
    eligible = master.loc[eligibility[year]].set_index('tract_id')
    original = eligible[EXTENDED_FEATURES].replace('', pd.NA).apply(pd.to_numeric, errors='raise').astype('Float64')
    result = original.copy()
    for source, cols in prepared_sources:
        part = source[year]
        part = part.set_index('tract_id') if 'tract_id' in part.columns else part
        assert part.index.is_unique and len(part) == len(eligible)
        assert set(part.index) == set(eligible.index), 'Input validation failed: check source values, donor availability, and expected record counts.'
        result.loc[:, cols] = part.reindex(eligible.index)[cols].astype('Float64')
    assert result.index.equals(eligible.index)
    assert result.notna().all().all()
    assert np.isfinite(result.to_numpy(dtype=float)).all()
    assert result.ge(0).all().all()
    for col in EXTENDED_FEATURES:
        obs = original[col].notna()
        pd.testing.assert_series_equal(result.loc[obs, col], original.loc[obs, col])
        field_fill_summary.append({'year': year, 'field': col, 'filled_cells': int((~obs).sum())})
    for col in ['clean__Urban', 'clean__GroupQuartersFlag', 'clean__LowIncomeTracts'] + SPATIAL_FIELDS[:3]:
        assert result[col].isin([0, 1]).all()
    for col in ['clean__PCTGQTRS', 'clean__PovertyRate']:
        assert result[col].between(0, 100).all()
    assert result.clean__population_total.gt(0).all()
    assert result[INCOME_COLUMN].gt(0).all()
    assert result[SPATIAL_FIELDS[3]].le(result.clean__population_total).all()
    race_cols = ['clean__Tract' + name for name in ['White', 'Black', 'Asian', 'NHOPI', 'AIAN', 'OMultir']]
    assert np.allclose(result[race_cols].sum(axis=1).to_numpy(dtype=float), result.clean__population_total.to_numpy(dtype=float))
    target = pd.to_numeric(eligible.FD_2, errors='raise').astype('int8').rename('FD_2')
    assert target.isin([0, 1]).all() and eligible.FD_2_status.eq('known').all()
    X_base[year] = result[BASE_FEATURES].copy()
    X_extended[year] = result[EXTENDED_FEATURES].copy()
    y_fd2[year] = target
    model_metadata[year] = eligible[['State', 'FD_2_status']].copy()
    model_metadata[year]['dataset_year'] = year
    model_metadata[year]['state_fips'] = eligible.index.str[:2]
    model_metadata[year]['county_fips'] = eligible.index.str[:5]
    model_metadata[year]['income_value_status'] = eligible.income_value_status
    imputed_flags[year] = original.isna().astype('int8').add_suffix('__imputed')
    assert target.index.equals(result.index) and model_metadata[year].index.equals(result.index)
    assert result.index.str.fullmatch('[0-9]{11}').all()
    assembly_summary.append({'year': year, 'rows': len(result), 'base_features': len(BASE_FEATURES), 'extended_features': len(EXTENDED_FEATURES), 'missing_X': int(result.isna().sum().sum()), 'missing_y': int(target.isna().sum()), 'filled_cells_base': int(original[BASE_FEATURES].isna().sum().sum()), 'filled_cells_extended': int(original.isna().sum().sum()), 'rows_with_any_fill': int(original.isna().any(axis=1).sum()), 'FD_2_0': int(target.eq(0).sum()), 'FD_2_1': int(target.eq(1).sum())})
assert X_base['2019'].columns.equals(X_base['2025'].columns)
assert X_extended['2019'].columns.equals(X_extended['2025'].columns)
assert len(X_extended['2019']) == 72527 and len(X_extended['2025']) == 83848
for year, path in source_paths.items():
    assert sha256(path) == source_hashes[year]
fill_summary_table = pd.DataFrame(field_fill_summary)

display(pd.DataFrame(assembly_summary))
display(pd.DataFrame(field_fill_summary).query('filled_cells > 0').reset_index(drop=True))
display(X_extended['2019'].head(3))


,year,rows,base_features,extended_features,missing_X,missing_y,filled_cells_base,filled_cells_extended,rows_with_any_fill,FD_2_0,FD_2_1
0,2019,72527,21,25,0,0,787,36697,36297,47818,24709
1,2025,83848,21,25,0,0,2348,2362,1873,70493,13355


,year,field,filled_cells
0,2019,clean__NUMGQTRS,21
1,2019,clean__PCTGQTRS,21
2,2019,income_with_endpoint_proxy_2018_usd,745
3,2019,clean__low_access_population_1_20,35910
4,2025,clean__PovertyRate,433
5,2025,income_with_endpoint_proxy_2018_usd,1873
6,2025,clean__TractLOWI,14
7,2025,clean__TractHUNV,14
8,2025,clean__TractSNAP,14
9,2025,clean__LATractsVehicle_20,14


,clean__Urban,clean__population_total,clean__occupied_housing_units_total,clean__GroupQuartersFlag,clean__NUMGQTRS,clean__PCTGQTRS,clean__TractKids,clean__TractSeniors,clean__TractWhite,clean__TractBlack,...,clean__LowIncomeTracts,clean__PovertyRate,income_with_endpoint_proxy_2018_usd,clean__TractLOWI,clean__TractHUNV,clean__TractSNAP,clean__LATracts_half,clean__LATracts1,clean__LATractsVehicle_20,clean__low_access_population_1_20
tract_id,,,,,,,,,,,,,,,,,,,,,
01001020100,1.0,1912.0,693.0,0.0,0.0,0.0,507.0,221.0,1622.0,217.0,...,0.0,11.3,81250.0,455.0,6.0,102.0,1.0,1.0,0.0,1896.0
01001020200,1.0,2170.0,743.0,0.0,181.0,8.34,606.0,214.0,888.0,1217.0,...,1.0,17.9,49000.0,802.0,89.0,156.0,1.0,1.0,0.0,1261.0
01001020300,1.0,3373.0,1256.0,0.0,0.0,0.0,894.0,439.0,2576.0,647.0,...,0.0,15.0,62609.0,1306.0,99.0,172.0,1.0,1.0,0.0,1552.0


## 24. Export the two model-input CSVs

Each CSV contains **27 columns: tract_id + 25 candidate features + FD_2**. The 21-feature option is selected using `BASE_FEATURES`; do not use every CSV column as X.

Every execution creates a new UTC-timestamped output directory under `DataCleaningFeatureEngeering/ProposalModelDataProcessing/deliverables/`. Existing files are not overwritten. Only the two data CSVs are written; no per-record logs or additional files are exported. The notebook retains the processing explanations and summary outputs.

CSV files are read back immediately to verify exact numeric values, IDs, feature order, targets, and absence of missing values. Retain tract IDs as strings when reading them. Each run with the same inputs should produce the same CSV contents even though the output directory name changes.


In [10]:
from datetime import datetime, timezone

output_dir = ROOT / 'DataCleaningFeatureEngeering/ProposalModelDataProcessing/deliverables' / (
    'model_inputs_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
output_dir.mkdir(parents=True, exist_ok=False)
export_summary = []
for year in ['2019', '2025']:
    ready = X_extended[year].copy()
    ready['FD_2'] = y_fd2[year]
    path = output_dir / f'model_input_{year}.csv'
    ready.to_csv(path, index=True, index_label='tract_id', encoding='utf-8-sig')
    # Python's float parser preserves the serialized float value across pandas versions.
    readback = pd.read_csv(path, dtype={'tract_id': str}, converters={c: float for c in EXTENDED_FEATURES}).set_index('tract_id')
    assert readback.index.equals(ready.index) and readback.index.is_unique
    assert readback.columns.tolist() == EXTENDED_FEATURES + ['FD_2']
    assert readback.notna().all().all()
    np.testing.assert_array_equal(readback[EXTENDED_FEATURES].to_numpy(dtype=float), ready[EXTENDED_FEATURES].to_numpy(dtype=float))
    np.testing.assert_array_equal(readback.FD_2.to_numpy(), y_fd2[year].to_numpy())
    export_summary.append({'year':year, 'rows':len(readback), 'columns_including_ID_and_target':len(readback.columns)+1,
                           'missing_cells':int(readback.isna().sum().sum()), 'round_trip_exact':True,
                           'csv_sha256':sha256(path)})
for year,path in source_paths.items():
    assert sha256(path) == source_hashes[year], 'A source CSV changed during execution.'
print('Output directory:', output_dir)
display(pd.DataFrame(export_summary))


Output directory: /Users/jennifer/Documents/ChatGPT/CSE6242/Project/cse6242-project/DataCleaningFeatureEngeering/ProposalModelDataProcessing/deliverables/model_inputs_20261102T012508074207Z


,year,rows,columns_including_ID_and_target,missing_cells,round_trip_exact,csv_sha256
0,2019,72527,27,0,True,7e9c8d11963aa0dfd2d295bf92679642ce879158d17965...
1,2025,83848,27,0,True,8f54ea037aac318c1c2c05d7dea1b22fa6a15b2b57d661...


## 25. Use the exported data

Select the same feature list for both years. Standardization, encoding, collinearity handling, missingness sensitivity analyses, and final model choices are downstream responsibilities. The CSVs are numerically complete candidates, not proof that all 25 features are appropriate together.

```python
train = pd.read_csv(output_dir / 'model_input_2019.csv', dtype={'tract_id': str}).set_index('tract_id')
test = pd.read_csv(output_dir / 'model_input_2025.csv', dtype={'tract_id': str}).set_index('tract_id')
features = BASE_FEATURES  # Or explicitly select EXTENDED_FEATURES after considering the spatial notes.
X_train, X_test = train[features], test[features]
y_train, y_test = train['FD_2'], test['FD_2']
```

For clustering, use only the selected X columns. If 2019 is the fitting year, applying that fitted clustering procedure to 2025 differs from independently clustering 2025. Interpretation of clusters or risk thresholds is outside this notebook.
